In [ ]:
import pandas as pd

# 1. 파일 불러오기
df = pd.read_csv('사료 데이터베이스 - 시트1.csv')

# 2. 1g당 칼로리 계산 함수
def calc_kcal_1g(row):
    # '%' 기호 제거 및 실수 변환
    prot_str = str(row['crude_protein']).replace('%', '').strip()
    fat_str = str(row['crude_fat']).replace('%', '').strip()
    
    try:
        prot = float(prot_str)
        fat = float(fat_str)
    except:
        prot, fat = 0.0, 0.0
        
    # 제형(건식/습식) 파악 (데이터가 main_protein 등에 섞여 있을 수 있어 합쳐서 검사)
    form_str = str(row['form']) + " " + str(row['main_protein'])
    
    if '습식' in form_str or '캔' in form_str or '파우치' in form_str:
        moisture, ash, fiber = 75, 2, 1
    else: # 그 외(스몰, 미디움 포함)는 건식 간주
        moisture, ash, fiber = 10, 7, 3
        
    # 탄수화물 (NFE) 계산
    nfe = 100 - (prot + fat + moisture + ash + fiber)
    if nfe < 0: nfe = 0
    
    # 100g당 칼로리 계산 후 1g 단위로 변환
    kcal_100g = (prot * 3.5) + (fat * 8.5) + (nfe * 3.5)
    return round(kcal_100g / 100.0, 2)

# 3. 데이터프레임에 함수 적용하여 'kcal' 열 채우기
df['kcal'] = df.apply(calc_kcal_1g, axis=1)

# 4. 결과 저장
df.to_csv('사료_데이터베이스_kcal_계산완료.csv', index=False, encoding='utf-8-sig')